In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Обучение BERT для расстановки ударений.

По сути, тут реализована до конца только часть классификатора. Сами функции замены нужных гласных на гласные со значком ударения не реализованы.

Код взят и адаптирован отсюда https://www.kaggle.com/code/mexwell/bert-for-binary-classification#Validation-Split

Я обучаю BERT multilingual cased.

In [ ]:
# копируем датасет с диска своего в рабочее место колаба
!cp /content/drive/MyDrive/Stress_vowel/data_for_backs.csv .

In [ ]:
# открываем датасет
import pandas as pd
df = pd.read_csv('data_for_backs.csv', sep = ',', encoding='utf8')
df.head()

,context,target_word,label
0,весь небосвод был облачен и хмур,облачен,1
1,"как был облачен тот осенний день, так и осталс...",облачен,1
2,Горизонт облачен и не сулит хорошей погоды,облачен,1
3,"Воздух влажен, и весь свод небесный облачен",облачен,1
4,"Если внутренний мир твой облачен, то и солнце ...",облачен,1


In [ ]:
df['stressed_vowel'] = [None for _ in range(len(df))]
df['jo'] = [0 for _ in range(len(df))]
df.head()

,context,target_word,label,stressed_vowel,jo
0,весь небосвод был облачен и хмур,облачен,1,None,0
1,"как был облачен тот осенний день, так и осталс...",облачен,1,None,0
2,Горизонт облачен и не сулит хорошей погоды,облачен,1,None,0
3,"Воздух влажен, и весь свод небесный облачен",облачен,1,None,0
4,"Если внутренний мир твой облачен, то и солнце ...",облачен,1,None,0


In [ ]:
df.label.max()

19

In [ ]:
# Сделаем ещё две колонки
# одна будет обозначать, на какой гласный по порядку падает ударение
# другая будет обозначать, является ли этот гласный Ё
first_syl = {1, 7, 8, 10, 12, 14, 18}
first_jo = {6}
second_syl = {3, 4, 9, 11, 13, 15, 19}
second_jo = {2}
third_syl = {5, 17}
third_jo = {0, 16}
for i in range(len(df)):
    if df.loc[i, 'label'] in first_syl:
        df.loc[i, 'stressed_vowel'] = 1
    elif df.loc[i, 'label'] in first_jo:
        df.loc[i, 'stressed_vowel'] = 1
        df.loc[i, 'jo'] = 1
    elif df.loc[i, 'label'] in second_syl:
        df.loc[i, 'stressed_vowel'] = 2
    elif df.loc[i, 'label'] in second_jo:
        df.loc[i, 'stressed_vowel'] = 2
        df.loc[i, 'jo'] = 1
    elif df.loc[i, 'label'] in third_syl:
        df.loc[i, 'stressed_vowel'] = 3
    elif df.loc[i, 'label'] in third_jo:
        df.loc[i, 'stressed_vowel'] = 3
        df.loc[i, 'jo'] = 1
df.head()

,context,target_word,label,stressed_vowel,jo
0,весь небосвод был облачен и хмур,облачен,1,1,0
1,"как был облачен тот осенний день, так и осталс...",облачен,1,1,0
2,Горизонт облачен и не сулит хорошей погоды,облачен,1,1,0
3,"Воздух влажен, и весь свод небесный облачен",облачен,1,1,0
4,"Если внутренний мир твой облачен, то и солнце ...",облачен,1,1,0


In [ ]:
df.tail()

,context,target_word,label,stressed_vowel,jo
385,"Я по одной мошке хватаю, а он ловит их сразу с...",мошке,18,1,0
386,Вот ведь и узники пустозерские в смрадном свое...,мошке,18,1,0
387,Летней мошке не расскажешь про лед - она огран...,мошке,19,2,0
388,"на окна поставить москитные сетки, чтобы устра...",мошке,19,2,0
389,"Там их привязали к столбам, обмазали их тела м...",мошке,19,2,0


In [ ]:
#а́е́и́о́у́ы́э́ю́я́
# пропишем функции для расстановки ударения по параметрам, чисто по приколу
import re
def set_emphasis(vowel:str, jo: int):
    '''Заменяет подаваемый гласный на него же с креской'''
    if jo==1:
        return 'ё'
    match vowel:
        case 'а': return 'а́'
        case 'е': return 'е́'
        case 'и': return 'и́'
        case 'о': return 'о́'
        case 'у': return 'у́'
        case 'ы': return 'ы́'
        case 'э': return 'э́'
        case 'ю': return 'ю́'
        case 'я': return 'я́'
def set_stress(word: str, label: int, stressed_vowel: int, jo: int):
    '''Принимает целевую словоформу и параметры, возвращает словоформу с поставленным ударением'''
    vowel_n_span = list(re.finditer(r'([аеиоуыэюя])', word))[stressed_vowel-1]
    emphasized_str = set_emphasis(vowel_n_span.group(), jo)
    span_ = vowel_n_span.span()[0]
    return word[:span_] + emphasized_str + word[span_+1:]

In [ ]:
# В принципе, это всё, зачем нам нужна эта функция
set_stress(*list(df.loc[100])[1:])

'нёба'

In [ ]:
# Чтобы модель обращала внимание именно на конкретное слово, надо добавить вокруг него специальный токен [TGT],
# а потом добавить его в словарь токенизатора и изменить длину эмбеддингов для модели
import re
tgtd_contexts = []
for i in range(len(df)):
  word = df.loc[i, 'target_word'].strip().lower() # на всяк случай стрип и ловер
  tgtd_context = re.sub(word, f'[TGT] {word} [TGT]', df.loc[i, 'context']) # добавляем токен
  if re.search('\[TGT\]', tgtd_context) is None: # если там целевое слово с большой буквы, ещё раз пробуем с капиталайзом
    tgtd_context = re.sub(word.capitalize(), f'[TGT] {word.capitalize()} [TGT]', df.loc[i, 'context'])
    if re.search('\[TGT\]', tgtd_context) is None:
      raise ValueError('Токен не был добавлен')
  tgtd_contexts.append(tgtd_context)
df['targeted_context'] = tgtd_contexts
df['targeted_context'].head()

<>:8: SyntaxWarning: invalid escape sequence '\['
<>:10: SyntaxWarning: invalid escape sequence '\['
<>:8: SyntaxWarning: invalid escape sequence '\['
<>:10: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_25628/2595454497.py:8: SyntaxWarning: invalid escape sequence '\['
  if re.search('\[TGT\]', tgtd_context) is None: # если там целевое слово с большой буквы, ещё раз пробуем с капиталайзом
/tmp/ipykernel_25628/2595454497.py:10: SyntaxWarning: invalid escape sequence '\['
  if re.search('\[TGT\]', tgtd_context) is None:


,targeted_context
0,весь небосвод был [TGT] облачен [TGT] и хмур
1,"как был [TGT] облачен [TGT] тот осенний день, ..."
2,Горизонт [TGT] облачен [TGT] и не сулит хороше...
3,"Воздух влажен, и весь свод небесный [TGT] обла..."
4,"Если внутренний мир твой [TGT] облачен [TGT], ..."


## Разбиение датасета

In [ ]:
# разделяй и обучай
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(df, train_size=0.8, shuffle=True, random_state=24)
print(f'Train size: {len(train_data)}', f'Test size: {len(test_data)}', sep = '\n')

Train size: 312
Test size: 78


In [ ]:
# Проверим, чтобы все лейблы были в обоих выборках
print(set(train_data['label']))
print(set(test_data['label']))

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19}
{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19}


In [ ]:
!pip install razdel

In [ ]:
# Сейчас надо прикинуть примерную максимальную длину контекстов в токенах, чтобы указать необходимое число в классе для датасетов
from razdel import tokenize
max_value = 0
for i in range(len(df)):
  tokens = list(tokenize(df.loc[i, 'context']))
  if len(tokens) > max_value:
    max_value = len(tokens)
print(max_value, 'Плюс ещё токен [TGT], и плюс ещё какая-то поправка, на всяк случай')

54 Плюс ещё токен [TGT], и плюс ещё какая-то поправка, на всяк случай


### Создадим класс для датасетов



In [ ]:
# Класс для датасетов
import torch
from transformers import AutoTokenizer
from torch.utils.data import Dataset
class TextDataset(Dataset):
  def __init__(self, dataframe, max_length=90, tokenizer_name='bert-base-multilingual-cased'):
    self.texts = dataframe['targeted_context'].values # Задаем нумпай вектор с контекстом
    self.targets = dataframe['label'].values # Задаем нумпай вектор с лейблами
    self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name) # токенизатор определяем
    self.tokenizer.add_tokens(['[TGT]']) # добавляем наш токен для выделения слова в токенизатор
    self.max_length = max_length

  def __len__(self):
    return len(self.texts)

  def __getitem__(self, idx):
    text = self.texts[idx]
    target = self.targets[idx]
    encoding = self.tokenizer(
        text=text,
        max_length=self.max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].flatten()
    attention_mask = encoding['attention_mask'].flatten()

    return {
        'input_ids': torch.as_tensor(input_ids, dtype=torch.long),
        'attention_mask': torch.as_tensor(attention_mask, dtype=torch.long),
        'label': torch.as_tensor(target, dtype=torch.long),
        'context': text
    }

In [ ]:
# делаем инстансы датасетов
train_dataset = TextDataset(train_data)
test_dataset = TextDataset(test_data)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# сделаем класс загрузчика данных, чтобы во время дообучения не надо было весь датасет подгружать сразу
# Но хотя он сам по себе небольшой, не знаю, пусть будет
from torch.utils.data import DataLoader

BATCH_SIZE = 16
torch.manual_seed(42)
train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True)
test_loader = DataLoader(test_dataset,
                         batch_size=len(test_dataset))
#next(iter(train_loader))

In [ ]:
# настраиваем девайс для обучения
from torch import cuda
torch.cuda.empty_cache()
device = 'cuda' if cuda.is_available() else 'cpu'

In [ ]:
# загружаем модель берта
from transformers import BertForSequenceClassification, BertConfig
from torch.optim import AdamW

model = BertForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels = 20, # у нас 20 классов, это важно не забывать делать
    output_attentions = False,
    output_hidden_states = False,
)
model.resize_token_embeddings(len(train_dataset.tokenizer)) # расширяем эмбеддинги, чтобы влез токен [TGT]
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119548, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [ ]:
# Гиперпараметры
# batch size уже задавали, он 16
EPOCHS = 15

optimizer = AdamW(model.parameters(),
                  lr = 1e-5,
                  eps = 1e-8)

In [ ]:
import time
import datetime
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
def save_checkpoint(epoch, model, optimizer, loss, filepath):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, filepath)
    print(f"Checkpoint saved at epoch {epoch} to {filepath}")

In [ ]:
# зададим функцию для чистки памяти
import gc
#from numba import cuda
#cuda.get_current_device().reset()
def clean_data(global_ = True):
  if global_:
    global model, optimizer, input_tensors
  try: del model
  except: pass
  try: del optimizer
  except: pass
  try: del input_tensors # delete any other tensors on GPU
  except: pass
  gc.collect() # Optional but recommended
  torch.cuda.empty_cache()

In [ ]:
# обучение модели
import os
def train_bert(state_):
    try:
        os.mkdir(f'BERT_stress{state_}')
    except:
        pass
    # готовим к трене
    training_stats = []
    epoch_loss_train = []
    total_t0 = time.time()

    # TRAINING
    for epoch in range(1, EPOCHS + 1):
        model.train()
        t0 = time.time()
        print("")
        print("================ Epoch {:} / {:} ================".format(epoch, EPOCHS))
        train_all_predictions = []
        train_all_true_labels = []
        for step, data in enumerate(train_loader):
            if step % 40 == 0 and not step == 0:
                elapsed = int(round(time.time() - t0))
                elapsed = str(datetime.timedelta(seconds=elapsed))
                print(
                    "  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.".format(
                        step, len(train_loader), elapsed
                    )
                )

            targets = data["label"].to(device)
            mask = data["attention_mask"].to(device)
            ids = data["input_ids"].to(device)

            model.zero_grad()

            loss, logits = model(
                ids, token_type_ids=None, attention_mask=mask, labels=targets
            ).to_tuple()
            epoch_loss_train.append(loss.item())

            cpu_logits = logits.cpu().detach().numpy()
            train_all_predictions.extend(np.argmax(cpu_logits, axis=1).flatten())
            train_all_true_labels.extend(targets.cpu().numpy())

            loss.backward()
            optimizer.step()
        train_accuracy = accuracy_score(train_all_true_labels, train_all_predictions)
        train_precision, train_recall, train_f1, _ = precision_recall_fscore_support(
            train_all_true_labels, train_all_predictions, average="macro"
        )
        print("")
        print('---TRAIN METRICS---')
        print(f"Loss: {np.mean(epoch_loss_train):.4f}")
        print(f"Accuracy: {train_accuracy:.4f}")
        print(f"Precision: {train_precision:.4f}")
        print(f"Recall: {train_recall:.4f}")
        print(f"F1-Score: {train_f1:.4f}")
        print("")
        #save_model
        state = {
            'epoch': epoch,
            'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict(),
        }
        save_checkpoint(epoch, model, optimizer, loss, f'/content/BERT_stress{state_}/checkpoint{epoch}.pth')
    clean_data(global_ = False)

In [ ]:
#валидация модели
def validate_bert(state_, EPOCHS = EPOCHS):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    test_stats = []
    for epoch in range(1, EPOCHS+1):
        print("================ Epoch {:} / {:} ================".format(epoch, EPOCHS))
        # loading model
        model = BertForSequenceClassification.from_pretrained(
        "bert-base-multilingual-cased",
        num_labels = 20, # двадцаточка, не забыть поставить
        output_attentions = False,
        output_hidden_states = False,
    )
        model.resize_token_embeddings(len(train_dataset.tokenizer)) # расширяем эмбеддинги
        checkpoint = torch.load(f"/content/BERT_stress{state_}/checkpoint{epoch}.pth", map_location=torch.device(device))
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        # VALIDATION
        print("Running validation ...")
        print("")
        model.eval()
        epoch_loss_test = []
        test_all_predictions = []
        test_all_true_labels = []
        for data in test_loader:
            targets = data["label"].to(device)
            mask = data["attention_mask"].to(device)
            ids = data["input_ids"].to(device)

            with torch.no_grad():
                loss, logits = model(ids, token_type_ids=None, attention_mask=mask, labels=targets).to_tuple()

            epoch_loss_test.append(loss.item())
            cpu_logits = logits.cpu().detach().numpy()
            test_all_predictions.extend(np.argmax(cpu_logits, axis=1).flatten())
            test_all_true_labels.extend(targets.cpu().numpy())
        test_accuracy = accuracy_score(test_all_true_labels, test_all_predictions)
        test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
            test_all_true_labels, test_all_predictions, average="macro"
        )
        print("")
        print('---TEST METRICS---')
        print(f"Loss: {np.mean(epoch_loss_test):.4f}")
        print(f"Accuracy: {test_accuracy:.4f}")
        print(f"Precision: {test_precision:.4f}")
        print(f"Recall: {test_recall:.4f}")
        print(f"F1-Score: {test_f1:.4f}")

        test_stats.append(
                {
                'epoch': epoch,
                'Loss': np.mean(epoch_loss_test),
                'Accuracy': test_accuracy,
                'Precision': test_precision,
                'Recall': test_recall,
                'F1-Score': test_f1
            }
        )
    clean_data(global_ = False)
    return test_stats

In [ ]:
#clean_data()

##Обучение для random_state == 24

In [ ]:
### Тут мы треним и тестим с random_state 24
# у нас уже датасет подготовлен с этим сидом, надо только нужные функции запустить
train_bert(24)
print('\n'*3)


================ Epoch 1 / 15 ================


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



---TRAIN METRICS---
Loss: 2.9844
Accuracy: 0.0513
Precision: 0.0357
Recall: 0.0448
F1-Score: 0.0347

Checkpoint saved at epoch 1 to /content/BERT_stress24/checkpoint1.pth

================ Epoch 2 / 15 ================


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



---TRAIN METRICS---
Loss: 2.9472
Accuracy: 0.0994
Precision: 0.1250
Recall: 0.0771
F1-Score: 0.0599

Checkpoint saved at epoch 2 to /content/BERT_stress24/checkpoint2.pth

================ Epoch 3 / 15 ================


...


---TRAIN METRICS---
Loss: 2.7365
Accuracy: 0.4359
Precision: 0.5361
Recall: 0.3776
F1-Score: 0.3484

Checkpoint saved at epoch 5 to /content/BERT_stress24/checkpoint5.pth

================ Epoch 6 / 15 ================


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



---TRAIN METRICS---
Loss: 2.6197
Accuracy: 0.6378
Precision: 0.7144
Recall: 0.5802
F1-Score: 0.5523

Checkpoint saved at epoch 6 to /content/BERT_stress24/checkpoint6.pth

================ Epoch 7 / 15 ================


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



---TRAIN METRICS---
Loss: 2.4962
Accuracy: 0.7019
Precision: 0.7710
Recall: 0.6511
F1-Score: 0.6343

Checkpoint saved at epoch 7 to /content/BERT_stress24/checkpoint7.pth

================ Epoch 8 / 15 ================

---TRAIN METRICS---
Loss: 2.3721
Accuracy: 0.8077
Precision: 0.8819
Recall: 0.7954
F1-Score: 0.7954

Checkpoint saved at epoch 8 to /content/BERT_stress24/checkpoint8.pth

================ Epoch 9 / 15 ================

---TRAIN METRICS---
Loss: 2.2516
Accuracy: 0.8429
Precision: 0.8859
Recall: 0.8267
F1-Score: 0.8330

Checkpoint saved at epoch 9 to /content/BERT_stress24/checkpoint9.pth

================ Epoch 10 / 15 ================

---TRAIN METRICS---
Loss: 2.1376
Accuracy: 0.9231
Precision: 0.9420
Recall: 0.9164
F1-Score: 0.9236

Checkpoint saved at epoch 10 to /content/BERT_stress24/checkpoint10.pth

================ Epoch 11 / 15 ================

---TRAIN METRICS---
Loss: 2.0300
Accuracy: 0.9295
Precision: 0.9429
Recall: 0.9203
F1-Score: 0.9260

Checkpoint sav

Да. 15 Эпох - это перебор

In [ ]:
clean_data()
state24 = validate_bert(24)

================ Epoch 1 / 15 ================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...


---TEST METRICS---
Loss: 0.8911
Accuracy: 0.7564
Precision: 0.8075
Recall: 0.8596
F1-Score: 0.7866
================ Epoch 15 / 15 ================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...


---TEST METRICS---
Loss: 0.7900
Accuracy: 0.8077
Precision: 0.8577
Recall: 0.8839
F1-Score: 0.8281


##Ну можно увидеть, что можно ещё несколько эпох навернуть.
Loss уменьшается, и другие показатели растут.
Поэтому напишем функцию для дообучения

In [ ]:
# Функция для дообучения
def finetune_bert(state_, last_epoch=15, epochs_to_finetune=5):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    # loading model
    model = BertForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels = 20, # двадцаточка, не забыть поставить
    output_attentions = False,
    output_hidden_states = False,
  )
    model.resize_token_embeddings(len(train_dataset.tokenizer)) # расширяем эмбеддинги
    checkpoint = torch.load(f"/content/BERT_stress{state_}/checkpoint{last_epoch}.pth", map_location=torch.device(device))
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    optimizer = AdamW(model.parameters(),
                  lr = 1e-5,
                  eps = 1e-8)
    training_stats = []
    epoch_loss_train = []
    total_t0 = time.time()
    # TRAINING
    for epoch in range(last_epoch, last_epoch + epochs_to_finetune + 1):
        model.train()
        t0 = time.time()
        print("")
        print("================ Epoch {:} / {:} ================".format(epoch, last_epoch+epochs_to_finetune))
        train_all_predictions = []
        train_all_true_labels = []
        for step, data in enumerate(train_loader):
            if step % 40 == 0 and not step == 0:
                elapsed = int(round(time.time() - t0))
                elapsed = str(datetime.timedelta(seconds=elapsed))
                print(
                    "  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.".format(
                        step, len(train_loader), elapsed
                    )
                )

            targets = data["label"].to(device)
            mask = data["attention_mask"].to(device)
            ids = data["input_ids"].to(device)

            model.zero_grad()

            loss, logits = model(
                ids, token_type_ids=None, attention_mask=mask, labels=targets
            ).to_tuple()
            epoch_loss_train.append(loss.item())

            cpu_logits = logits.cpu().detach().numpy()
            train_all_predictions.extend(np.argmax(cpu_logits, axis=1).flatten())
            train_all_true_labels.extend(targets.cpu().numpy())

            loss.backward()
            optimizer.step()
        train_accuracy = accuracy_score(train_all_true_labels, train_all_predictions)
        train_precision, train_recall, train_f1, _ = precision_recall_fscore_support(
            train_all_true_labels, train_all_predictions, average="macro"
        )
        print("")
        print('---TRAIN METRICS---')
        print(f"Loss: {np.mean(epoch_loss_train):.4f}")
        print(f"Accuracy: {train_accuracy:.4f}")
        print(f"Precision: {train_precision:.4f}")
        print(f"Recall: {train_recall:.4f}")
        print(f"F1-Score: {train_f1:.4f}")
        print("")
        #save_model
        state = {
            'epoch': epoch,
            'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict(),
        }
        save_checkpoint(epoch, model, optimizer, loss, f'/content/BERT_stress{state_}/checkpoint{epoch}.pth')

In [ ]:
# Дообучение
clean_data()
finetune_bert(24)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



================ Epoch 15 / 18 ================

---TRAIN METRICS---
Loss: 0.4329
Accuracy: 0.9968
Precision: 0.9964
Recall: 0.9938
F1-Score: 0.9948

Checkpoint saved at epoch 15 to /content/BERT_stress24/checkpoint15.pth

================ Epoch 16 / 18 ================

---TRAIN METRICS---
Loss: 0.3916
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1-Score: 1.0000

Checkpoint saved at epoch 16 to /content/BERT_stress24/checkpoint16.pth

================ Epoch 17 / 18 ================

---TRAIN METRICS---
Loss: 0.3584
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1-Score: 1.0000

Checkpoint saved at epoch 17 to /content/BERT_stress24/checkpoint17.pth

================ Epoch 18 / 18 ================

---TRAIN METRICS---
Loss: 0.3279
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1-Score: 1.0000

Checkpoint saved at epoch 18 to /content/BERT_stress24/checkpoint18.pth
================ Epoch 1 / 15 ================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...


---TEST METRICS---
Loss: 0.8911
Accuracy: 0.7564
Precision: 0.8075
Recall: 0.8596
F1-Score: 0.7866
================ Epoch 15 / 15 ================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...


---TEST METRICS---
Loss: 0.6941
Accuracy: 0.8205
Precision: 0.8643
Recall: 0.9005
F1-Score: 0.8317


In [ ]:
clean_data()
state24 = validate_bert(24, 18)
clean_data()

================ Epoch 1 / 18 ================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



---TEST METRICS---
Loss: 0.6310
Accuracy: 0.8590
Precision: 0.8567
Recall: 0.8857
F1-Score: 0.8359


## Обучение для random_state == 2

In [ ]:
### Треня для random_state 2
train_data, test_data = train_test_split(df, train_size=0.8, shuffle=True, random_state=2)
print(f'Train size: {len(train_data)}', f'Test size: {len(test_data)}', sep = '\n')
# делаем инстансы датасетов
train_dataset = TextDataset(train_data)
test_dataset = TextDataset(test_data)
# Создаем загрузчик
BATCH_SIZE = 16
torch.manual_seed(42)
train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True)
test_loader = DataLoader(test_dataset,
                         batch_size=len(test_dataset))
# Загрузка модели берта
model = BertForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels = 20, # у нас 20 классов
    output_attentions = False,
    output_hidden_states = False,
)
model.resize_token_embeddings(len(train_dataset.tokenizer)) # расширяем эмбеддинги, чтобы влез токен [TGT]
model.to(device)
# Задаем эпохи и оптимайзер
EPOCHS = 18
optimizer = AdamW(model.parameters(),
                  lr = 1e-5,
                  eps = 1e-8)
# Обучение и проверка
train_bert(2)
clean_data()
state2 = validate_bert(2)
clean_data()

Train size: 312
Test size: 78


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



================ Epoch 1 / 18 ================


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



---TEST METRICS---
Loss: 0.8991
Accuracy: 0.7179
Precision: 0.7881
Recall: 0.8250
F1-Score: 0.7827


In [ ]:
clean_data()
state2 = validate_bert(2, 18)
clean_data()

================ Epoch 1 / 18 ================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...


---TEST METRICS---
Loss: 0.7912
Accuracy: 0.7564
Precision: 0.8648
Recall: 0.8577
F1-Score: 0.8314
================ Epoch 18 / 18 ================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...


---TEST METRICS---
Loss: 0.7356
Accuracy: 0.7692
Precision: 0.8517
Recall: 0.8490
F1-Score: 0.8155


## Обучение для random_state == 22

In [ ]:
### Треня для random_state 22
clean_data()
train_data, test_data = train_test_split(df, train_size=0.8, shuffle=True, random_state=22)
print(f'Train size: {len(train_data)}', f'Test size: {len(test_data)}', sep = '\n')
# делаем инстансы датасетов
train_dataset = TextDataset(train_data)
test_dataset = TextDataset(test_data)
# Создаем загрузчик
BATCH_SIZE = 16
torch.manual_seed(42)
train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True)
test_loader = DataLoader(test_dataset,
                         batch_size=len(test_dataset))
# Загрузка модели берта
model = BertForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels = 20, # у нас 20 классов
    output_attentions = False,
    output_hidden_states = False,
)
model.resize_token_embeddings(len(train_dataset.tokenizer)) # расширяем эмбеддинги, чтобы влез токен [TGT]
model.to(device)
# Задаем эпохи и оптимайзер
EPOCHS = 18
optimizer = AdamW(model.parameters(),
                  lr = 1e-5,
                  eps = 1e-8)
# Обучение и проверка
train_bert(22)

Train size: 312
Test size: 78


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



================ Epoch 1 / 18 ================


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...


---TRAIN METRICS---
Loss: 2.8083
Accuracy: 0.4359
Precision: 0.6025
Recall: 0.3764
F1-Score: 0.3606

Checkpoint saved at epoch 6 to /content/BERT_stress22/checkpoint6.pth

================ Epoch 7 / 18 ================


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



---TRAIN METRICS---
Loss: 2.7062
Accuracy: 0.6218
Precision: 0.6737
Recall: 0.5654
F1-Score: 0.5284

Checkpoint saved at epoch 7 to /content/BERT_stress22/checkpoint7.pth

================ Epoch 8 / 18 ================


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



---TRAIN METRICS---
Loss: 2.5918
Accuracy: 0.7372
Precision: 0.7643
Recall: 0.7002
F1-Score: 0.6863

Checkpoint saved at epoch 8 to /content/BERT_stress22/checkpoint8.pth

================ Epoch 9 / 18 ================

---TRAIN METRICS---
Loss: 2.4734
Accuracy: 0.7949
Precision: 0.8462
Recall: 0.7788
F1-Score: 0.7832

Checkpoint saved at epoch 9 to /content/BERT_stress22/checkpoint9.pth

================ Epoch 10 / 18 ================

---TRAIN METRICS---
Loss: 2.3551
Accuracy: 0.8878
Precision: 0.9065
Recall: 0.8760
F1-Score: 0.8805

Checkpoint saved at epoch 10 to /content/BERT_stress22/checkpoint10.pth

================ Epoch 11 / 18 ================

---TRAIN METRICS---
Loss: 2.2398
Accuracy: 0.9487
Precision: 0.9552
Recall: 0.9454
F1-Score: 0.9477

Checkpoint saved at epoch 11 to /content/BERT_stress22/checkpoint11.pth

================ Epoch 12 / 18 ================

---TRAIN METRICS---
Loss: 2.1313
Accuracy: 0.9551
Precision: 0.9588
Recall: 0.9510
F1-Score: 0.9521

Checkpoint 

In [ ]:
clean_data()
state22 = validate_bert(22, 18)
clean_data()

================ Epoch 1 / 18 ================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...


---TEST METRICS---
Loss: 0.6906
Accuracy: 0.8333
Precision: 0.8819
Recall: 0.8339
F1-Score: 0.8360
================ Epoch 18 / 18 ================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Running validation ...


---TEST METRICS---
Loss: 0.6603
Accuracy: 0.8462
Precision: 0.8877
Recall: 0.8714
F1-Score: 0.8649


##Сравним результаты на разных эпохах
Как по мне, на 18 эпохе самый удачный результат везде.

Тогда измерим среднее accuracy для всех сидов по 18 эпохе.

In [ ]:
# сравним три разные выборки
print(state24[-1]['Accuracy'], state2[-1]['Accuracy'], state22[-1]['Accuracy'], sep = '\n')
mean_accuracy = (state24[-1]['Accuracy']+state2[-1]['Accuracy']+state22[-1]['Accuracy'])/3
print(f'Mean accuracy is:{mean_accuracy:.4f}')

0.8589743589743589
0.7692307692307693
0.8461538461538461
Mean accuracy is:0.8248


In [ ]:
# Сохраним это в файл
data = {"rs24": state24, "rs2": state2, "rs22": state22}
import json
with open('all_seeds_epochs_metrics.json', 'w', encoding='utf8') as f:
  json.dump(data, f, ensure_ascii=False, indent=4)

## Заключение
Вероятно, можно было бы обучать модель ещё несколько эпох и это повысило бы accuracy.

Я понял, что мне следовало бы сделать learning rate побольше, это бы уменьшило количество эпох.

P.S. Каждая новая эпоха как додеп.

In [ ]:
import requests

In [ ]:
requests.get('https://zws2-1.web.telegram.org/apiw1')